In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('source/csv/FReDA3.csv')

In [ ]:
df = df.rename(
    columns={
        'Anchor Health': 'Focal General Health',
        'Partner Health': 'Partner General Health',

        "Anchor Neuroticism": "Focal Neuroticism",
        "Anchor Extraversion": "Focal Extraversion",
        "Anchor Openness": "Focal Openness",
        "Anchor Agreeableness": "Focal Agreeableness",
        "Anchor Conscientiousness": "Focal Conscientiousness",
        "Anchor Depressiveness": "Focal Depressiveness",
        "Anchor Loneliness": "Focal Loneliness",
        "Anchor Self-esteem": "Focal Self-esteem",
        "Anchor Life Satisfaction": "Focal Life Satisfaction",
        "Anchor Religiosity": "Focal Religiosity",
        "Anchor Conservatism": "Focal Conservatism",
        "Anchor Relationship Satisfaction": "Focal Relationship Satisfaction",
        "Anchor Communication Quality": "Focal Communication Quality",
        "Anchor Conflict Management": "Focal Conflict Management",
    }
)

In [ ]:
# 1. (1 = Satisfied, 0 = All others)
df['Focal Is_Satisfied'] = np.where(df['Anchor Perception'] == 'Satisfied', 1, 0)
df['Partner Is_Satisfied'] = np.where(df['Partner Perception'] == 'Satisfied', 1, 0)

If we build personaltiy categories (age-corrected quartiles), how do they relate to touch?

In [ ]:
df["Focal Age_Quartile"] = pd.qcut(df["Anchor Age"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
df["Partner Age_Quartile"] = pd.qcut(df["Partner Age"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

In [ ]:
df["Focal Relationship Satisfaction"].quantile([0.25, 0.50, 0.75, 0.95])

In [ ]:
def assign_quartiles(x):
    q25 = x.quantile(0.25)
    q50 = x.quantile(0.50)
    q75 = x.quantile(0.75)

    # 2. Define conditions
    conditions = [
        (x <= q25),
        (x > q25) & (x <= q50),
        (x > q50) & (x <= q75),
        (x > q75)
    ]

    labels = ["Q1", "Q2", "Q3", "Q4"]

    result_array = np.select(conditions, labels, default="NaN_temp")

    return pd.Series(result_array, index=x.index).replace("NaN_temp", np.nan)

In [ ]:
# df['Extraversion_q'] = df.groupby('Age_Quartile', observed=True)['Extraversion'].transform(assign_quartiles)

traits = [
    "Focal Neuroticism",
    "Focal Extraversion",
    "Focal Openness",
    "Focal Agreeableness",
    "Focal Conscientiousness",
    "Focal Depressiveness",
    "Focal Loneliness",
    "Focal Self-esteem",
    "Focal Life Satisfaction",
    "Focal General Health",
    "Focal Religiosity",
    "Focal Conservatism",
    "Focal Relationship Satisfaction",
    "Focal Communication Quality",
    "Focal Conflict Management",
]
new_columns = [f"{trait}_q" for trait in traits]
df[new_columns] = df.groupby('Focal Age_Quartile', observed=True)[traits].transform(assign_quartiles)

In [ ]:
traits = [
    "Partner Neuroticism",
    "Partner Extraversion",
    "Partner Openness",
    "Partner Agreeableness",
    "Partner Conscientiousness",
    "Partner Depressiveness",
    "Partner Loneliness",
    "Partner Self-esteem",
    "Partner Life Satisfaction",
    "Partner General Health",
    "Partner Religiosity",
    "Partner Conservatism",
    "Partner Relationship Satisfaction",
    "Partner Communication Quality",
    "Partner Conflict Management",
]
new_columns = [f"{trait}_q" for trait in traits]
df[new_columns] = df.groupby('Partner Age_Quartile', observed=True)[traits].transform(assign_quartiles)

If I am in the upper quatile of extraversion, how likely is it that I am satisfied with touch, how likely is it that my partner is satisfied with touch?

In [ ]:
# Extract only upper quartile (above75%) df[df['Extraversion_q'] == 'Q4']
# Calculate mean of satisfied

for trait in traits:
    print(f"{trait}_q")
    prob_satisfied = df[df[f"{trait}_q"] == 'Q4']['Focal Is_Satisfied'].mean()
    print(prob_satisfied)

In [ ]:
quartiles = ["Q1", "Q2", "Q3", "Q4"]
for trait in traits:
    print(f"{trait}_q")
    for q in quartiles:
            prob_satisfied = df[df[f"{trait}_q"] == q]['Focal Is_Satisfied'].mean()
            print(f"Likelihood of Satisfaction for {q}: {prob_satisfied:.1%}")

In [ ]:
prob_anchor4 = df[df["Focal Neuroticism_q"] == "Q4"]['Focal Is_Satisfied'].mean()
prob_partner4 = df[df["Focal Neuroticism_q"] == "Q4"]['Partner Is_Satisfied'].mean()

print("If I am in Q4 of Extraversion")
print(f"Likelihood of Satisfaction (Me) {prob_anchor4:.1%}")
print(f"Likelihood of Satisfaction (Partner) {prob_partner4:.1%}")

prob_anchor1 = df[df["Focal Neuroticism_q"] == "Q1"]['Focal Is_Satisfied'].mean()
prob_partner1 = df[df["Focal Neuroticism_q"] == "Q1"]['Partner Is_Satisfied'].mean()

print("If I am in Q1 of Extraversion")
print(f"Likelihood of Satisfaction (Me) {prob_anchor1:.1%}")
print(f"Likelihood of Satisfaction (Partner) {prob_partner1:.1%}")

In [ ]:
prob_anchor4 = df[df["Focal Loneliness_q"] == "Q4"]['Focal Is_Satisfied'].mean()
prob_partner4 = df[df["Focal Loneliness_q"] == "Q4"]['Partner Is_Satisfied'].mean()

print("If I am in Q4 of Loneliness")
print(f"Likelihood of Satisfaction (Me) {prob_anchor4:.1%}")
print(f"Likelihood of Satisfaction (Partner) {prob_partner4:.1%}")

prob_anchor1 = df[df["Focal Loneliness_q"] == "Q1"]['Focal Is_Satisfied'].mean()
prob_partner1 = df[df["Focal Loneliness_q"] == "Q1"]['Partner Is_Satisfied'].mean()

print("If I am in Q1 of Loneliness")
print(f"Likelihood of Satisfaction (Me) {prob_anchor1:.1%}")
print(f"Likelihood of Satisfaction (Partner) {prob_partner1:.1%}")

If I am in the upper quartile of estraversion + the upper quartile of agreeableness, how likely is it that my partner is satisfied with touch?

In [ ]:
prob_anchor4 = df[(df["Focal Loneliness_q"] == "Q1") & (df["Focal Extraversion_q"] == "Q4")]['Focal Is_Satisfied'].mean()
prob_partner4 = df[(df["Focal Loneliness_q"] == "Q1") & (df["Focal Extraversion_q"] == "Q4")]['Partner Is_Satisfied'].mean()

print("If I am in Q4 of Loneliness and Q4 of Extraversion")
print(f"Likelihood of Satisfaction (Me) {prob_anchor4:.1%}")
print(f"Likelihood of Satisfaction (Partner) {prob_partner4:.1%}")

And so on

And what happens to touch if both partners are in the upper quartile of extraversion etc?

In [ ]:
# q1 = [
#     df[(df["Focal Loneliness_q"] == "Q1") & (df["Focal Extraversion_q"] == "Q1")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q2") & (df["Focal Extraversion_q"] == "Q1")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q3") & (df["Focal Extraversion_q"] == "Q1")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q4") & (df["Focal Extraversion_q"] == "Q1")]['Focal Is_Satisfied'].mean(),
# ]
#
# q2 = [
#     df[(df["Focal Loneliness_q"] == "Q1") & (df["Focal Extraversion_q"] == "Q2")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q2") & (df["Focal Extraversion_q"] == "Q2")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q3") & (df["Focal Extraversion_q"] == "Q2")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q4") & (df["Focal Extraversion_q"] == "Q2")]['Focal Is_Satisfied'].mean(),
# ]
#
# q3 = [
#     df[(df["Focal Loneliness_q"] == "Q1") & (df["Focal Extraversion_q"] == "Q3")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q2") & (df["Focal Extraversion_q"] == "Q3")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q3") & (df["Focal Extraversion_q"] == "Q3")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q4") & (df["Focal Extraversion_q"] == "Q3")]['Focal Is_Satisfied'].mean(),
# ]
#
# q4 = [
#     df[(df["Focal Loneliness_q"] == "Q1") & (df["Focal Extraversion_q"] == "Q4")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q2") & (df["Focal Extraversion_q"] == "Q4")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q3") & (df["Focal Extraversion_q"] == "Q4")]['Focal Is_Satisfied'].mean(),
#     df[(df["Focal Loneliness_q"] == "Q4") & (df["Focal Extraversion_q"] == "Q4")]['Focal Is_Satisfied'].mean(),
# ]

In [ ]:
test_df_focal = df.groupby(["Focal Loneliness_q", "Partner Loneliness_q"])["Focal Is_Satisfied"].mean().reset_index()

test_df_focal.rename(columns={
    "Focal Loneliness_q": "Focal Loneliness",
    "Partner Loneliness_q": "Partner Loneliness",
    "Focal Is_Satisfied": "Focal Satisfaction"
}, inplace=True)

In [ ]:
pal = [
    '#7DA28B',
    '#5D8AA8',
    '#D4A373',
    '#C0778B'
]

sns.lineplot(
    data=test_df_focal,
    x="Focal Loneliness",
    y="Focal Satisfaction",
    hue="Partner Loneliness",
    palette=pal,
    marker="o",
    markersize=8,
    linewidth=2,
)

plt.ylabel("Focal Satisfaction", fontsize=14)
plt.xlabel("Focal Loneliness", fontsize=14)

plt.ylim(0.2, 0.8)
plt.grid(False)
plt.grid(axis='y', linestyle='--', alpha=0.5, color='gray')
plt.show()

In [ ]:
test_df_partner = df.groupby(["Focal Loneliness_q", "Partner Loneliness_q"])["Partner Is_Satisfied"].mean().reset_index()

test_df_partner.rename(columns={
    "Focal Loneliness_q": "Focal Loneliness",
    "Partner Loneliness_q": "Partner Loneliness",
    "Partner Is_Satisfied": "Partner Satisfaction"
}, inplace=True)

In [ ]:
pal = [
    '#7DA28B',
    '#5D8AA8',
    '#D4A373',
    '#C0778B'
]

sns.lineplot(
    data=test_df_partner,
    x="Focal Loneliness",
    y="Partner Satisfaction",
    hue="Partner Loneliness",
    palette=pal,
    marker="o",
    markersize=8,
    linewidth=2,
)

plt.ylabel("Partner Satisfaction", fontsize=14)
plt.xlabel("Focal Loneliness", fontsize=14)

plt.ylim(0.2, 0.8)
plt.grid(False)
plt.grid(axis='y', linestyle='--', alpha=0.5, color='gray')
plt.show()

In [ ]:
def plot_traits(input_df, trait1, trait2, target, count):
    plot_df = input_df.groupby([trait1, trait2])[target].mean().reset_index()
    clean_trait1 = trait1.replace("Focal ", "").replace("_q", "")
    clean_trait2 = trait2.replace("Focal ", "").replace("_q", "")

    plot_df.rename(columns={
        trait1: clean_trait1,
        trait2: clean_trait2,
        target: "Satisfaction"
    }, inplace=True)

    pal = {
        "Q1": "#5D8AA8",
        "Q2": "#D4A373",
        "Q3": "#C0778B",
        "Q4": "#7DA28B"
    }

    sns.lineplot(
        data=plot_df,
        x=clean_trait1,
        y="Satisfaction",
        hue=clean_trait2,
        palette=pal,
        marker="o",
        markersize=8,
        linewidth=2,
    )

    plt.ylabel("Satisfaction", fontsize=14)
    plt.xlabel(clean_trait1, fontsize=14)

    plt.ylim(0.2, 0.8)
    plt.grid(False)
    plt.grid(axis='y', linestyle='--', alpha=0.5, color='gray')

    output_folder = "output/img/quartiles/"
    os.makedirs(output_folder, exist_ok=True)

    filename = os.path.join(output_folder, f"{clean_trait2}.png")

    plt.savefig(filename, dpi=600, bbox_inches='tight')

    # plt.show()
    print(f"Printing {count}/14")
    plt.close()

In [ ]:
target1 = "Focal Extraversion_q"
target2 = "Partner Is_Satisfied"

for i, trait in enumerate(traits):
    trait = f"{trait}_q"
    if trait != target1:
        plot_traits(df, target1, trait, target2, i)
    else:
        print(
            f"Skipping {trait.replace('Focal ', '').replace('_q', '')}"
        )